# ChromaDB Parent-Child Chunk Health Check

Validates the integrity of parent-child chunk relationships stored in ChromaDB.

This notebook performs:
- Connection and collection verification
- Child and parent chunk enumeration
- Orphan detection (children without parents, parents without children)
- Metadata consistency validation
- Relationship health scoring
- Detailed remediation recommendations

## 0. Import Required Libraries

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from chromadb import PersistentClient
from scripts.ingest.ingest_config import IngestConfig
import pandas as pd
import json
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional

print("✓ Libraries imported successfully")

## 1. Connect to ChromaDB

In [ ]:
# Load configuration
from scripts.utils.db_factory import get_default_vector_path, get_vector_client
from scripts.ingest.ingest_config import IngestConfig
from pathlib import Path

config = IngestConfig()

# Get the correct ChromaDB path using factory pattern
PersistentClient_class, using_sqlite = get_vector_client(prefer="chroma")
chroma_path = get_default_vector_path(Path(config.rag_data_path), using_sqlite)

print(f"rag_data_path: {config.rag_data_path}")
print(f"ChromaDB path: {chroma_path}")
print()

# Connect to ChromaDB
client = PersistentClient_class(path=str(chroma_path))

print(f"✓ Connected to ChromaDB at: {chroma_path}")
print()

# List available collections
available_collections = client.list_collections()
print(f"Available collections ({len(available_collections)}):")
for col in available_collections:
    print(f"  - {col.name} (count: {col.count()})")

print()

# Try to get chunk collection if it exists
try:
    chunk_collection = client.get_collection(config.chunk_collection_name)
    print(f"✓ Chunk collection: {config.chunk_collection_name}")
    print(f"✓ Collection count: {chunk_collection.count()}")
except Exception as e:
    print(f"⚠ Chunk collection not found: {config.chunk_collection_name}")
    print(f"  Error: {e}")
    chunk_collection = None

## 2. Query Child Entries

In [ ]:
# Retrieve all chunks
if chunk_collection is None:
    print("⚠ Chunk collection not available - skipping this section")
    all_chunks = None
else:
    all_chunks = chunk_collection.get(
        include=["documents", "metadatas"]
    )

if all_chunks:
    # Separate child and parent chunks
    child_chunks = []
    parent_chunks = []
    regular_chunks = []

    for chunk_id, metadata in zip(all_chunks["ids"], all_chunks["metadatas"]):
        chunk_data = {
            "id": chunk_id,
            "metadata": metadata,
            "is_parent": metadata.get("is_parent", False) if metadata else False,
            "chunk_type": metadata.get("chunk_type", "regular") if metadata else "regular"
        }
        
        if chunk_data["is_parent"]:
            parent_chunks.append(chunk_data)
        elif chunk_data["chunk_type"] == "child" or "parent_id" in (metadata or {}):
            child_chunks.append(chunk_data)

## 3. Query Parent Entries

In [ ]:
# Build parent ID -> child IDs mapping
parent_to_children: Dict[str, Set[str]] = defaultdict(set)
child_to_parent: Dict[str, str] = {}

for child in child_chunks:
    parent_id = child["metadata"].get("parent_id") if child["metadata"] else None
    if parent_id:
        parent_to_children[parent_id].add(child["id"])
        child_to_parent[child["id"]] = parent_id

# Parent chunk analysis
print(f"📊 Parent chunk analysis:")
print(f"  - Total parent chunks stored: {len(parent_chunks)}")
print(f"  - Parents with children references: {len(parent_to_children)}")

# List parent chunks with their child counts
print(f"\n📋 Parent chunks and their children:")
for parent in parent_chunks[:10]:
    parent_id = parent["id"]
    child_count = len(parent_to_children.get(parent_id, set()))
    
    # Try to parse child_ids from metadata
    child_ids_json = parent["metadata"].get("child_ids") if parent["metadata"] else None
    if child_ids_json:
        try:
            child_ids_list = json.loads(child_ids_json)
            stored_child_count = len(child_ids_list)
            print(f"  {parent_id}")
            print(f"    - Stored child_ids in metadata: {stored_child_count}")
            print(f"    - Actual children found: {child_count}")
        except json.JSONDecodeError:
            print(f"  {parent_id} - Could not parse child_ids metadata")

## 4. Validate Parent-Child Relationships

In [ ]:
# Validate relationships
issues = {
    "orphaned_children": [],  # Children without parents
    "orphaned_parents": [],   # Parents without children
    "missing_parent_records": [],  # Children referencing non-existent parents
    "mismatched_child_counts": []  # Stored vs actual child counts
}

# Check for orphaned children
for child in child_chunks:
    parent_id = child["metadata"].get("parent_id") if child["metadata"] else None
    
    if not parent_id:
        issues["orphaned_children"].append(child["id"])
    else:
        # Check if parent exists
        parent_exists = any(p["id"] == parent_id for p in parent_chunks)
        if not parent_exists:
            issues["missing_parent_records"].append({
                "child_id": child["id"],
                "referenced_parent_id": parent_id
            })

# Check for orphaned parents
all_parent_ids = {p["id"] for p in parent_chunks}
for parent_id in all_parent_ids:
    if parent_id not in parent_to_children or len(parent_to_children[parent_id]) == 0:
        issues["orphaned_parents"].append(parent_id)

# Check for mismatched child counts
for parent in parent_chunks:
    parent_id = parent["id"]
    child_ids_json = parent["metadata"].get("child_ids") if parent["metadata"] else None
    
    if child_ids_json:
        try:
            stored_child_ids = json.loads(child_ids_json)
            actual_child_ids = parent_to_children.get(parent_id, set())
            
            if len(stored_child_ids) != len(actual_child_ids):
                issues["mismatched_child_counts"].append({
                    "parent_id": parent_id,
                    "stored_count": len(stored_child_ids),
                    "actual_count": len(actual_child_ids)
                })
        except json.JSONDecodeError:
            pass

print("🔍 Relationship Validation Results:")
print(f"  - Orphaned children (no parent_id): {len(issues['orphaned_children'])}")
print(f"  - Orphaned parents (no children): {len(issues['orphaned_parents'])}")
print(f"  - Missing parent records: {len(issues['missing_parent_records'])}")
print(f"  - Mismatched child counts: {len(issues['mismatched_child_counts'])}")

# Show details of issues
if issues["orphaned_children"]:
    print(f"\n⚠️  Orphaned children (first 5):")
    for child_id in issues["orphaned_children"][:5]:
        print(f"  - {child_id}")

if issues["missing_parent_records"]:
    print(f"\n⚠️  Missing parent records (first 5):")
    for issue in issues["missing_parent_records"][:5]:
        print(f"  - Child {issue['child_id']} references missing parent {issue['referenced_parent_id']}")

if issues["orphaned_parents"]:
    print(f"\n⚠️  Orphaned parents (first 5):")
    for parent_id in issues["orphaned_parents"][:5]:
        print(f"  - {parent_id}")

if issues["mismatched_child_counts"]:
    print(f"\n⚠️  Mismatched child counts (first 5):")
    for issue in issues["mismatched_child_counts"][:5]:
        print(f"  - Parent {issue['parent_id']}: stored={issue['stored_count']}, actual={issue['actual_count']}")

## 5. Generate Health Check Report

In [ ]:
# Calculate health score
total_issues = sum(len(v) if isinstance(v, list) else 0 for v in issues.values())
issue_count = sum(1 for v in issues.values() if (isinstance(v, list) and len(v) > 0) or (isinstance(v, int) and v > 0))

# Health score: 100 - (issues * 10)
health_score = max(0, 100 - (total_issues * 2))

print("=" * 70)
print("📊 CHROMADB PARENT-CHILD CHUNK HEALTH CHECK REPORT")
print("=" * 70)

print(f"\n🎯 Overall Health Score: {health_score}/100", end="")
if health_score >= 90:
    print(" ✅ EXCELLENT")
elif health_score >= 70:
    print(" ⚠️  GOOD")
elif health_score >= 50:
    print(" ⚠️  FAIR")
else:
    print(" ❌ POOR")

print(f"\n📈 Collection Statistics:")
print(f"  - Total chunks: {len(all_chunks['ids'])}")
print(f"  - Parent chunks: {len(parent_chunks)}")
print(f"  - Child chunks: {len(child_chunks)}")
print(f"  - Regular chunks: {len(regular_chunks)}")

print(f"\n🔗 Relationship Statistics:")
print(f"  - Parents with children: {len(parent_to_children)}")
print(f"  - Children with parents: {len(child_to_parent)}")
if len(parent_chunks) > 0:
    avg_children_per_parent = len(child_chunks) / len(parent_chunks) if len(parent_chunks) > 0 else 0
    print(f"  - Average children per parent: {avg_children_per_parent:.1f}")

print(f"\n⚠️  Issues Found: {total_issues}")
print(f"  - Orphaned children: {len(issues['orphaned_children'])}")
print(f"  - Orphaned parents: {len(issues['orphaned_parents'])}")
print(f"  - Missing parent records: {len(issues['missing_parent_records'])}")
print(f"  - Mismatched child counts: {len(issues['mismatched_child_counts'])}")

# Recommendations
print(f"\n💡 Recommendations:")
if health_score >= 90:
    print("  ✅ Database is healthy. No action required.")
elif len(issues["orphaned_children"]) > 0:
    print(f"  ⚠️  Consider re-ingesting {len(issues['orphaned_children'])} orphaned children")
elif len(issues["orphaned_parents"]) > 0:
    print(f"  ⚠️  Consider cleaning up {len(issues['orphaned_parents'])} unused parent chunks")
elif len(issues["missing_parent_records"]) > 0:
    print(f"  ⚠️  Found {len(issues['missing_parent_records'])} broken relationships - re-ingestion recommended")
else:
    print("  ✅ No critical issues detected.")

print("\n" + "=" * 70)

## 6. Export Summary as DataFrame

In [ ]:
# Create summary dataframes for export

# Summary of chunk types
chunk_summary = pd.DataFrame({
    "Type": ["Parent", "Child", "Regular", "Total"],
    "Count": [len(parent_chunks), len(child_chunks), len(regular_chunks), len(all_chunks['ids'])],
    "Percentage": [
        f"{len(parent_chunks)/len(all_chunks['ids'])*100:.1f}%",
        f"{len(child_chunks)/len(all_chunks['ids'])*100:.1f}%",
        f"{len(regular_chunks)/len(all_chunks['ids'])*100:.1f}%",
        "100%"
    ]
})

# Summary of issues
issue_summary = pd.DataFrame({
    "Issue Type": ["Orphaned Children", "Orphaned Parents", "Missing Parent Records", "Mismatched Child Counts"],
    "Count": [
        len(issues["orphaned_children"]),
        len(issues["orphaned_parents"]),
        len(issues["missing_parent_records"]),
        len(issues["mismatched_child_counts"])
    ],
    "Severity": ["Medium", "Low", "High", "Medium"]
})

print("\n📋 Chunk Type Distribution:")
print(chunk_summary.to_string(index=False))

print("\n\n⚠️  Issue Summary:")
print(issue_summary.to_string(index=False))

# Parent-Child mapping summary
parent_child_summary = []
for parent in parent_chunks:
    parent_id = parent["id"]
    children = parent_to_children.get(parent_id, set())
    
    parent_child_summary.append({
        "Parent ID": parent_id,
        "Child Count": len(children),
        "Doc ID": parent["metadata"].get("doc_id", "N/A") if parent["metadata"] else "N/A",
        "Version": parent["metadata"].get("version", "N/A") if parent["metadata"] else "N/A"
    })

parent_child_df = pd.DataFrame(parent_child_summary)

if len(parent_child_df) > 0:
    print("\n\n📊 Parent-Child Distribution (first 10):")
    print(parent_child_df.head(10).to_string(index=False))
    
    if len(parent_child_df) > 10:
        print(f"   ... and {len(parent_child_df) - 10} more parent chunks")